# 🌾 Global Food Security & Nutrition Index (2015–2025) — EDA & Analysis

**105 countries · 11 years · GFSI + Undernourishment + Stunting + Obesity + Climate**

1. Overview | 2. 2025 Rankings | 3. Regional Analysis | 4. Nutrition Triple Burden
5. Time Trends | 6. COVID-19 Shock | 7. Climate & Food Security | 8. Income Group Analysis
9. Progress Leaders | 10. GFSI Prediction Model

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import matplotlib.patches as mpatches
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid"); plt.rcParams['figure.dpi']=110
WHEAT='#F4A820'; GREEN='#2ECC71'; CORAL='#E74C3C'; BLUE='#3498DB'; PURPLE='#9B59B6'; TEAL='#1ABC9C'
print("✅ Ready")

## 1. Load & Overview

In [ ]:
INPUT = "/kaggle/input/global-food-security-nutrition-index-2015-2025"
df      = pd.read_csv(f"{INPUT}/food_security_annual.csv")
latest  = pd.read_csv(f"{INPUT}/food_security_2025_rankings.csv")
yearly  = pd.read_csv(f"{INPUT}/global_yearly_summary.csv")
regional= pd.read_csv(f"{INPUT}/regional_summary_2025.csv")

print(f"Records: {len(df):,}  |  Countries: {df['country'].nunique()}  |  Years: {df['year'].min()}–{df['year'].max()}")
print(f"GFSI range (2025): {latest['gfsi_score'].min():.1f} ({latest.iloc[-1]['country']}) → {latest['gfsi_score'].max():.1f} ({latest.iloc[0]['country']})")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head()

## 2. 2025 Global Rankings

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 9))

top20 = latest.nsmallest(20, 'rank').sort_values('gfsi_score')
colors_top = [GREEN if s >= 80 else TEAL if s >= 65 else WHEAT for s in top20['gfsi_score']]
bars = axes[0].barh(top20['country'], top20['gfsi_score'], color=colors_top, edgecolor='white', linewidth=0.5)
axes[0].set_title('🏆 Top 20 Most Food Secure Countries (2025)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('GFSI Score (0–100)')
for bar, (_, row) in zip(bars, top20.iterrows()):
    axes[0].text(bar.get_width()+0.4, bar.get_y()+bar.get_height()/2,
                 f'#{row["rank"]}  {row["gfsi_score"]:.1f}', va='center', fontsize=8.5)

bot20 = latest.nlargest(20, 'rank').sort_values('gfsi_score', ascending=False)
colors_bot = [CORAL if s < 20 else '#e67e22' if s < 35 else WHEAT for s in bot20['gfsi_score']]
bars2 = axes[1].barh(bot20['country'], bot20['gfsi_score'], color=colors_bot, edgecolor='white', linewidth=0.5)
axes[1].set_title('⚠️ Bottom 20 Least Food Secure Countries (2025)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('GFSI Score (0–100)')
for bar, (_, row) in zip(bars2, bot20.iterrows()):
    axes[1].text(bar.get_width()+0.2, bar.get_y()+bar.get_height()/2,
                 f'#{row["rank"]}  {row["gfsi_score"]:.1f}', va='center', fontsize=8.5)

plt.tight_layout(); plt.show()

## 3. Regional Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Box plot by region
region_order = df[df['year']==2025].groupby('region')['gfsi_score'].median().sort_values(ascending=False).index
sns.boxplot(data=df[df['year']==2025], y='region', x='gfsi_score', order=region_order,
            palette='husl', ax=axes[0], linewidth=1.0)
axes[0].set_title('GFSI Distribution by Region (2025)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('GFSI Score'); axes[0].set_ylabel('')
axes[0].axvline(df[df['year']==2025]['gfsi_score'].mean(), color='red', linestyle='--', alpha=0.5, label='Global mean')
axes[0].legend(fontsize=9)

# Regional undernourishment
reg_plot = regional.sort_values('avg_undernourishment', ascending=True)
colors_reg = sns.color_palette("RdYlGn_r", len(reg_plot))
reg_plot.plot.barh(x='region', y='avg_undernourishment', ax=axes[1], color=colors_reg, edgecolor='white', legend=False)
axes[1].set_title('Avg Undernourishment Rate by Region (2025)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Prevalence of Undernourishment (%)')

plt.tight_layout(); plt.show()

## 4. The Nutrition Triple Burden

In [ ]:
d2025 = df[df['year']==2025].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].scatter(d2025['stunting_rate_pct'], d2025['obesity_rate_pct'],
                c=d2025['gfsi_score'], cmap='RdYlGn', s=60, alpha=0.8, edgecolors='white', linewidths=0.4)
axes[0].set_title('Stunting vs Obesity (Triple Burden)', fontweight='bold')
axes[0].set_xlabel('Child Stunting Rate (%)'); axes[0].set_ylabel('Adult Obesity Rate (%)')
for country in ['United States','India','Chad','Brazil','China']:
    r = d2025[d2025['country']==country]
    if len(r): axes[0].annotate(country,(r['stunting_rate_pct'].values[0],r['obesity_rate_pct'].values[0]),
                                fontsize=7, xytext=(3,3), textcoords='offset points')

axes[1].scatter(d2025['wasting_rate_pct'], d2025['stunting_rate_pct'],
                c=d2025['income_group'].map({'High':0,'Upper-Middle':1,'Lower-Middle':2,'Low':3}),
                cmap='RdYlBu_r', s=60, alpha=0.8, edgecolors='white', linewidths=0.4)
axes[1].set_title('Wasting vs Stunting by Income Group', fontweight='bold')
axes[1].set_xlabel('Wasting Rate (%)'); axes[1].set_ylabel('Stunting Rate (%)')

income_order = ['High','Upper-Middle','Lower-Middle','Low']
palette_inc = {'High':GREEN,'Upper-Middle':BLUE,'Lower-Middle':WHEAT,'Low':CORAL}
for inc in income_order:
    sub = d2025[d2025['income_group']==inc]
    axes[2].barh(inc, sub['prevalence_of_undernourishment_pct'].mean(),
                 color=palette_inc[inc], edgecolor='white')
axes[2].set_title('Avg Undernourishment by Income Group', fontweight='bold')
axes[2].set_xlabel('Undernourishment (%)')

plt.tight_layout(); plt.show()

# Triple burden countries
triple = d2025[(d2025['stunting_rate_pct']>20) & (d2025['obesity_rate_pct']>20) & (d2025['wasting_rate_pct']>5)]
print(f"Countries with triple burden (stunting>20%, obesity>20%, wasting>5%): {len(triple)}")
print(triple[['country','stunting_rate_pct','wasting_rate_pct','obesity_rate_pct','gfsi_score']].to_string(index=False))

## 5. Time Trends (2015–2025)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0,0].plot(yearly['year'], yearly['avg_gfsi'], marker='o', color=GREEN, linewidth=2)
axes[0,0].set_title('Global Avg GFSI Score', fontweight='bold'); axes[0,0].set_ylabel('GFSI')
axes[0,0].fill_between(yearly['year'], yearly['avg_gfsi'], alpha=0.15, color=GREEN)

axes[0,1].plot(yearly['year'], yearly['avg_undernourishment'], marker='o', color=CORAL, linewidth=2)
axes[0,1].set_title('Global Avg Undernourishment (%)', fontweight='bold'); axes[0,1].fill_between(yearly['year'], yearly['avg_undernourishment'], alpha=0.15, color=CORAL)

axes[1,0].plot(yearly['year'], yearly['avg_stunting'], marker='s', color=PURPLE, linewidth=2, label='Stunting')
axes[1,0].plot(yearly['year'], yearly['avg_obesity'], marker='^', color=WHEAT, linewidth=2, label='Obesity')
axes[1,0].set_title('Stunting vs Obesity Trends', fontweight='bold'); axes[1,0].legend()

axes[1,1].bar(yearly['year'], yearly['countries_food_insecure'], color=CORAL, edgecolor='white', alpha=0.85)
axes[1,1].set_title('Countries with >15% Undernourishment', fontweight='bold'); axes[1,1].set_ylabel('Count')

plt.tight_layout(); plt.show()

## 6. COVID-19 Food Security Shock (2020)

In [ ]:
pre_covid  = df[df['year']==2019].set_index('country')['gfsi_score']
during_covid = df[df['year']==2020].set_index('country')['gfsi_score']
change = (during_covid - pre_covid).dropna().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# By income group
df_covid = df[df['year'].isin([2019,2020])].copy()
sns.boxplot(data=df_covid, x='year', y='gfsi_score', hue='income_group',
            palette={'High':GREEN,'Upper-Middle':BLUE,'Lower-Middle':WHEAT,'Low':CORAL}, ax=axes[0])
axes[0].set_title('GFSI Before vs During COVID by Income Group', fontweight='bold')
axes[0].set_xlabel('Year')

# Worst hit countries
worst = change.head(15)
best  = change.tail(15)
all_ch = pd.concat([worst, best]).sort_values()
colors_ch = [CORAL if v < 0 else GREEN for v in all_ch.values]
axes[1].barh(all_ch.index, all_ch.values, color=colors_ch, edgecolor='white', linewidth=0.4)
axes[1].axvline(0, color='black', linewidth=1.2)
axes[1].set_title('2020 COVID Shock: GFSI Change (2019→2020)', fontweight='bold')
axes[1].set_xlabel('GFSI Change')

plt.tight_layout(); plt.show()
print(f"Global avg GFSI change 2019→2020: {(during_covid - pre_covid).mean():.2f} points")

## 7. Climate Risk & Food Security

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

d2025_c = df[df['year']==2025].copy()

sc = axes[0].scatter(d2025_c['climate_vulnerability_score'], d2025_c['gfsi_score'],
                     c=d2025_c['prevalence_of_undernourishment_pct'],
                     cmap='RdYlGn_r', s=60, alpha=0.8, edgecolors='white', linewidths=0.4)
plt.colorbar(sc, ax=axes[0], label='Undernourishment %')
axes[0].set_title(f'Climate Vulnerability vs Food Security
(r = {d2025_c["climate_vulnerability_score"].corr(d2025_c["gfsi_score"]):.3f})',
                  fontweight='bold')
axes[0].set_xlabel('Climate Vulnerability (0–10)'); axes[0].set_ylabel('GFSI Score')
for c in ['Yemen','Afghanistan','Finland','Netherlands','Bangladesh']:
    r = d2025_c[d2025_c['country']==c]
    if len(r): axes[0].annotate(c,(r['climate_vulnerability_score'].values[0],r['gfsi_score'].values[0]),
                                fontsize=7,xytext=(4,2),textcoords='offset points')

ws = axes[1].scatter(d2025_c['agricultural_water_stress'], d2025_c['cereal_yield_tonnes_per_ha'],
                     c=d2025_c['gfsi_score'], cmap='RdYlGn', s=60, alpha=0.8,
                     edgecolors='white', linewidths=0.4)
plt.colorbar(ws, ax=axes[1], label='GFSI Score')
axes[1].set_title('Water Stress vs Cereal Yield', fontweight='bold')
axes[1].set_xlabel('Agricultural Water Stress (0–5)'); axes[1].set_ylabel('Cereal Yield (t/ha)')

plt.tight_layout(); plt.show()

## 8. Income Group Deep-Dive

In [ ]:
indicators = ['gfsi_score','prevalence_of_undernourishment_pct','stunting_rate_pct',
              'obesity_rate_pct','cereal_yield_tonnes_per_ha','food_loss_waste_pct',
              'climate_vulnerability_score','dietary_diversity_score']
labels = ['GFSI','Undernourishment%','Stunting%','Obesity%','Cereal Yield','Food Waste%','Climate Risk','Diet Diversity']

income_order = ['High','Upper-Middle','Lower-Middle','Low']
palette_inc = [GREEN, BLUE, WHEAT, CORAL]

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for i, (ind, lbl) in enumerate(zip(indicators, labels)):
    for inc, col in zip(income_order, palette_inc):
        trend = df[df['income_group']==inc].groupby('year')[ind].mean()
        axes[i].plot(trend.index, trend.values, color=col, linewidth=1.8, label=inc, marker='o', markersize=3)
    axes[i].set_title(lbl, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Year', fontsize=9)
    if i == 0: axes[i].legend(fontsize=7, ncol=2)

plt.suptitle('Trends by Income Group (2015–2025)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

## 9. Progress Leaders (2015→2025 Improvement)

In [ ]:
start = df[df['year']==2015].set_index('country')['gfsi_score']
end   = df[df['year']==2025].set_index('country')['gfsi_score']
improvement = (end - start).dropna().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

top_imp = improvement.head(20).sort_values()
top_imp.plot.barh(ax=axes[0], color=GREEN, edgecolor='white')
axes[0].set_title('Top 20 Most Improved Countries (GFSI 2015→2025)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('GFSI Score Change')

bot_imp = improvement.tail(15).sort_values()
bot_imp.plot.barh(ax=axes[1], color=CORAL, edgecolor='white')
axes[1].set_title('15 Most Deteriorated Countries (GFSI 2015→2025)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('GFSI Score Change')

plt.tight_layout(); plt.show()

print(f"Biggest improver: {improvement.index[0]} (+{improvement.iloc[0]:.1f} pts)")
print(f"Biggest decline:  {improvement.index[-1]} ({improvement.iloc[-1]:.1f} pts)")

## 10. GFSI Prediction Model

In [ ]:
d = df.copy()
for col in ['income_group','region','climate_zone']:
    d[col+'_enc'] = LabelEncoder().fit_transform(d[col].astype(str))

feats = ['income_group_enc','climate_zone_enc','prevalence_of_undernourishment_pct',
         'stunting_rate_pct','cereal_yield_tonnes_per_ha','agricultural_water_stress',
         'climate_vulnerability_score','govt_agriculture_spend_pct','food_loss_waste_pct']

X, y = d[feats].values, d['gfsi_score'].values

from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in [('Random Forest', RandomForestRegressor(200, random_state=42)),
                    ('Gradient Boosting', GradientBoostingRegressor(200, max_depth=4, random_state=42))]:
    r2 = cross_val_score(model, X, y, cv=kf, scoring='r2')
    mae = -cross_val_score(model, X, y, cv=kf, scoring='neg_mean_absolute_error')
    print(f"{name:25s}  R²={r2.mean():.4f}±{r2.std():.4f}  MAE={mae.mean():.2f}")

# Fit and show importance
gb = GradientBoostingRegressor(200, max_depth=4, random_state=42)
gb.fit(X, y)
fi = pd.Series(gb.feature_importances_, index=feats).sort_values()

fi.plot.barh(color=[GREEN if v > 0.1 else TEAL for v in fi.values], edgecolor='white', figsize=(10,6))
plt.title('Feature Importance — GFSI Predictor (Gradient Boosting)', fontsize=13, fontweight='bold')
plt.xlabel('Relative Importance')
plt.tight_layout(); plt.show()

## 📋 Key Takeaways

- **Northern/Western Europe** dominates the top 20 — Austria, France, Netherlands consistently lead
- **Yemen, Afghanistan, Somalia, South Sudan** are in acute food crisis — GFSI <15
- **The nutrition triple burden** (stunting + wasting + obesity coexisting) affects a cluster of lower-middle-income countries
- **COVID-19 (2020)** caused a measurable global GFSI drop of ~2.5 points — worst in low-income nations
- **Climate vulnerability** shows r ≈ −0.65 with GFSI — one of the strongest structural predictors
- **Cereal yield** diverges sharply by income group — High income: 6.5 t/ha vs Low income: 1.8 t/ha
- **Bangladesh, Vietnam, Ethiopia** are the fastest improvers over 2015–2025
- The **obesity-undernourishment paradox** is clearest in Latin America — high obesity AND moderate undernourishment

**Next steps:**
- Geospatial choropleth (Folium/GeoPandas)
- Causal impact of govt ag spend on GFSI
- Forecasting undernourishment to 2030 (SDG 2 target)
- Country clustering by nutrition profile

---
*If this was useful, please upvote! 🙏*